In [3]:
#!/usr/bin/env python3
"""
Replot manuscript Figure 4B-G from minimal aggregated plotting data,
using the original Fig. 4 layout.

"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator


# ============================================================
# Paths
# ============================================================

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

MINIMAL_FILE = SCRIPT_DIR / "fig4_plotdata.parquet"

OUT_PDF = SCRIPT_DIR / "figure4_BG.pdf"
OUT_PNG = SCRIPT_DIR / "figure4_BG.png"
OUT_SVG = SCRIPT_DIR / "figure4_BG.svg"


# ============================================================
# Original-style figure settings
# ============================================================

VAX_COLOR = "#4e79a7"
UNVAX_COLOR = "#e15759"

STATUS_COLOR = {
    "Unvaccinated": UNVAX_COLOR,
    "Vaccinated": VAX_COLOR,
}

N_HIST_BINS = 41
VAR_MAX_X = 0.20

# Original-style inset positions
WORK_INSET_CONTENT_SHIFT_X = 0.012

# Slightly moved right relative to the old [0.020, 0.505, 0.500, 0.455]
# but still using the same original family-inset logic.
FAMILY_CORR_INSET_RECT = [0.055, 0.505, 0.500, 0.455]


plt.rcParams.update({
    "font.family":      "sans-serif",
    "font.sans-serif":  ["Helvetica", "Arial", "DejaVu Sans"],

    "font.size":        20,
    "axes.titlesize":   20,
    "axes.labelsize":   20,
    "xtick.labelsize":  18,
    "ytick.labelsize":  18,
    "legend.fontsize":  17,

    "axes.spines.top":  False,
    "axes.spines.right": False,
    "axes.grid":        False,
    "axes.linewidth":   1.05,

    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size":  4.8,
    "ytick.major.size":  4.8,

    "legend.frameon":    False,
    "figure.dpi":        160,
    "savefig.dpi":       600,

    "pdf.fonttype":      42,
    "ps.fonttype":       42,
    "svg.fonttype":      "none",
})


# ============================================================
# Main histogram plotting from minimal data
# ============================================================

def clean_axis(ax, nbins_y: int = 5) -> None:
    ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins_y))
    ax.tick_params(axis="both", which="major", pad=3)


def draw_binned_density(
    ax,
    plotdata: pd.DataFrame,
    panel: str,
    variable: str,
    title: str,
    xlab: str,
    shade: bool = False,
) -> None:
    if shade:
        ax.set_facecolor("#f1f1f1")

    sub = plotdata[
        (plotdata["row_type"] == "main_hist")
        & (plotdata["panel"] == panel)
        & (plotdata["variable"] == variable)
    ].copy()

    if sub.empty:
        raise ValueError(f"No main histogram data for panel={panel}, variable={variable}")

    for status in ["Unvaccinated", "Vaccinated"]:
        s = sub[sub["status"] == status].sort_values("bin_left")
        if s.empty:
            continue

        ax.bar(
            s["bin_left"].to_numpy(dtype=float),
            s["density"].to_numpy(dtype=float),
            width=s["bin_width"].to_numpy(dtype=float),
            align="edge",
            color=STATUS_COLOR[status],
            alpha=0.65,
            edgecolor="white",
            linewidth=0.45,
            label=status,
        )

    ax.set_xlim(float(sub["bin_left"].min()), float(sub["bin_right"].max()))
    ax.set_title(title, loc="left", fontweight="bold", fontsize=20, pad=9)
    ax.set_xlabel(xlab)
    ax.set_ylabel("Density")
    clean_axis(ax)


def add_vaccination_status_legend_to_education(ax) -> None:
    handles = [
        Patch(facecolor=UNVAX_COLOR, alpha=0.80, label="Unvaccinated"),
        Patch(facecolor=VAX_COLOR, alpha=0.80, label="Vaccinated"),
    ]

    ax.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.50, 0.985),
        ncol=1,
        frameon=False,
        handlelength=1.6,
        labelspacing=0.25,
        borderaxespad=0.25,
    )


# ============================================================
# Work inset: original layout, but drawn from minimal data
# ============================================================

def draw_work_exact_inset(ax, plotdata: pd.DataFrame, c: int, show_xlabel: bool = False) -> None:
    sub = plotdata[
        (plotdata["row_type"] == "inset_exact")
        & (plotdata["layer"] == "work")
        & (plotdata["group_size"] == float(c))
    ].copy()

    if sub.empty:
        ax.text(
            0.5, 0.5, "empty",
            transform=ax.transAxes,
            ha="center", va="center",
            fontsize=11, color="0.5",
        )
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(rf"$c={c}$", fontsize=14, pad=4)
        if show_xlabel:
            ax.set_xlabel("Vaccinated neighbors", fontsize=14, labelpad=2)
        return

    x = np.arange(c)
    w = 0.38

    for status, offset in [("Unvaccinated", -w / 2), ("Vaccinated", w / 2)]:
        s = sub[sub["status"] == status].sort_values("x_value")

        ax.bar(
            s["x_value"].to_numpy(dtype=float) + offset,
            s["probability"].to_numpy(dtype=float),
            width=w,
            color=STATUS_COLOR[status],
            alpha=0.80,
            edgecolor="white",
            linewidth=0.30,
        )

    ax.set_xlim(-0.65, (c - 1) + 0.65)
    ax.set_title(rf"$c={c}$", fontsize=14, pad=4)

    if c <= 3:
        ax.set_xticks(np.arange(c))
    else:
        ax.set_xticks([0, c - 1])

    if show_xlabel:
        ax.set_xlabel("Vaccinated neighbors", fontsize=14, labelpad=2)

    ax.yaxis.set_major_locator(MaxNLocator(nbins=2))
    ax.tick_params(axis="both", labelsize=12, pad=2, length=2.5)


def draw_work_6plus_inset(ax, plotdata: pd.DataFrame) -> None:
    sub = plotdata[
        (plotdata["row_type"] == "inset_fraction_hist")
        & (plotdata["layer"] == "work")
        & (plotdata["size_bin"] == "c>=6")
    ].copy()

    if sub.empty:
        ax.text(
            0.5, 0.5, "empty",
            transform=ax.transAxes,
            ha="center", va="center",
            fontsize=11, color="0.5",
        )
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(r"$c\geq 6$", fontsize=14, pad=4)
        ax.set_xlabel("Vaccinated fraction", fontsize=14, labelpad=2)
        return

    for status in ["Unvaccinated", "Vaccinated"]:
        s = sub[sub["status"] == status].sort_values("bin_left")
        if s.empty:
            continue

        ax.bar(
            s["bin_left"].to_numpy(dtype=float),
            s["density"].to_numpy(dtype=float),
            width=s["bin_width"].to_numpy(dtype=float),
            align="edge",
            color=STATUS_COLOR[status],
            alpha=0.65,
            edgecolor="white",
            linewidth=0.30,
        )

    ax.set_xlim(0, 1)
    ax.set_title(r"$c\geq 6$", fontsize=14, pad=4)
    ax.set_xticks([0, 0.5, 1])
    ax.set_xlabel("Vaccinated fraction", fontsize=14, labelpad=2)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=2))
    ax.tick_params(axis="both", labelsize=12, pad=2, length=2.5)


def plot_work_composition_inset(ax_parent, plotdata: pd.DataFrame) -> None:
    """
    Original successful work-inset layout copied from the original notebook.
    """
    sx = WORK_INSET_CONTENT_SHIFT_X

    rects = {
        2:       [0.060 + sx, 0.755, 0.205, 0.160],
        3:       [0.305 + sx, 0.755, 0.205, 0.160],
        4:       [0.550 + sx, 0.755, 0.205, 0.160],
        5:       [0.060 + sx, 0.470, 0.205, 0.160],
        "6plus": [0.305 + sx, 0.470, 0.450, 0.160],
    }

    frame = Rectangle(
        (0.020, 0.320),
        0.770,
        0.655,
        transform=ax_parent.transAxes,
        facecolor="white",
        edgecolor="#666666",
        linewidth=0.9,
        zorder=2.5,
        clip_on=False,
    )
    ax_parent.add_patch(frame)

    for c in [2, 3, 4]:
        ax = ax_parent.inset_axes(rects[c], transform=ax_parent.transAxes)
        ax.set_facecolor("white")
        draw_work_exact_inset(ax, plotdata, c, show_xlabel=False)

    ax5 = ax_parent.inset_axes(rects[5], transform=ax_parent.transAxes)
    ax5.set_facecolor("white")
    draw_work_exact_inset(ax5, plotdata, 5, show_xlabel=True)

    ax6 = ax_parent.inset_axes(rects["6plus"], transform=ax_parent.transAxes)
    ax6.set_facecolor("white")
    draw_work_6plus_inset(ax6, plotdata)


# ============================================================
# Household panel: original GridSpec layout, minimal data
# ============================================================

def draw_household_exact(ax, plotdata: pd.DataFrame, c: int, show_ylabel: bool, show_xlabel: bool = True) -> None:
    sub = plotdata[
        (plotdata["row_type"] == "inset_exact")
        & (plotdata["layer"] == "household")
        & (plotdata["group_size"] == float(c))
    ].copy()

    if sub.empty:
        ax.text(
            0.5, 0.5, "empty",
            transform=ax.transAxes,
            ha="center", va="center",
            fontsize=14, color="0.5",
        )
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(rf"$c={c}$", fontsize=15, pad=5)
        return

    x = np.arange(c)
    w = 0.38

    for status, offset in [("Unvaccinated", -w / 2), ("Vaccinated", w / 2)]:
        s = sub[sub["status"] == status].sort_values("x_value")

        ax.bar(
            s["x_value"].to_numpy(dtype=float) + offset,
            s["probability"].to_numpy(dtype=float),
            width=w,
            color=STATUS_COLOR[status],
            alpha=0.80,
            edgecolor="white",
            linewidth=0.35,
        )

    ax.set_title(rf"$c={c}$", fontsize=15, pad=5)
    ax.set_xlim(-0.65, (c - 1) + 0.65)
    ax.set_xticks(np.arange(c))

    if show_ylabel:
        ax.set_ylabel("Density", fontsize=15, labelpad=2)
    else:
        ax.tick_params(labelleft=False)

    if show_xlabel:
        ax.set_xlabel("Vaccinated neighbors", fontsize=15, labelpad=2)

    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    ax.tick_params(axis="both", labelsize=13, pad=2, length=3)


def draw_household_6plus(ax, plotdata: pd.DataFrame) -> None:
    sub = plotdata[
        (plotdata["row_type"] == "inset_fraction_hist")
        & (plotdata["layer"] == "household")
        & (plotdata["size_bin"] == "c>=6")
    ].copy()

    if sub.empty:
        ax.text(
            0.5, 0.5, "empty",
            transform=ax.transAxes,
            ha="center", va="center",
            fontsize=14, color="0.5",
        )
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(r"$c\geq 6$", fontsize=15, pad=5)
        return

    for status in ["Unvaccinated", "Vaccinated"]:
        s = sub[sub["status"] == status].sort_values("bin_left")
        if s.empty:
            continue

        ax.bar(
            s["bin_left"].to_numpy(dtype=float),
            s["density"].to_numpy(dtype=float),
            width=s["bin_width"].to_numpy(dtype=float),
            align="edge",
            color=STATUS_COLOR[status],
            alpha=0.65,
            edgecolor="white",
            linewidth=0.35,
        )

    ax.set_xlim(0, 1)
    ax.set_title(r"$c\geq 6$", fontsize=15, pad=5)
    ax.set_xticks([0, 0.5, 1])
    ax.set_xlabel("Vaccinated fraction", fontsize=15, labelpad=2)
    ax.tick_params(labelleft=False)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    ax.tick_params(axis="both", labelsize=13, pad=2, length=3)


def plot_household_panel_binned(fig, gs_slot, plotdata: pd.DataFrame, panel_title: str = "D. Household") -> None:
    sub = GridSpecFromSubplotSpec(
        3,
        3,
        subplot_spec=gs_slot,
        height_ratios=[0.55, 1.0, 1.0],
        width_ratios=[1.0, 1.0, 1.35],
        hspace=1.45,
        wspace=0.55,
    )

    ax_title = fig.add_subplot(sub[0, :])
    ax_title.axis("off")

    ax_title.set_title(
        panel_title,
        loc="left",
        fontweight="bold",
        fontsize=20,
        pad=11,
    )

    ax2 = fig.add_subplot(sub[1, 0])
    draw_household_exact(ax2, plotdata, c=2, show_ylabel=True, show_xlabel=False)

    ax3 = fig.add_subplot(sub[1, 1])
    draw_household_exact(ax3, plotdata, c=3, show_ylabel=False, show_xlabel=False)

    ax4 = fig.add_subplot(sub[1, 2])
    draw_household_exact(ax4, plotdata, c=4, show_ylabel=False, show_xlabel=False)

    ax5 = fig.add_subplot(sub[2, 0])
    draw_household_exact(ax5, plotdata, c=5, show_ylabel=True, show_xlabel=True)

    ax6 = fig.add_subplot(sub[2, 1:3])
    draw_household_6plus(ax6, plotdata)


# ============================================================
# Family correlation inset: original logic, minimal data
# ============================================================

def plot_family_correlation_inset(ax_parent, plotdata: pd.DataFrame) -> None:
    sub = plotdata[plotdata["row_type"] == "family_corr"].copy()

    if sub.empty:
        return

    sub = sub.sort_values("order", ascending=False).reset_index(drop=True)
    y = np.arange(len(sub))

    obs = sub["r_observed"].to_numpy(dtype=float)
    obs_lo = sub["r_bootstrap_low95"].to_numpy(dtype=float)
    obs_hi = sub["r_bootstrap_high95"].to_numpy(dtype=float)

    null_mu = sub["r_global_shuffle_mean"].to_numpy(dtype=float)
    null_lo = sub["r_global_shuffle_low95"].to_numpy(dtype=float)
    null_hi = sub["r_global_shuffle_high95"].to_numpy(dtype=float)

    ax = ax_parent.inset_axes(
        FAMILY_CORR_INSET_RECT,
        transform=ax_parent.transAxes,
        zorder=10,
    )

    ax.set_facecolor("white")
    ax.patch.set_alpha(0.985)

    ax.errorbar(
        null_mu,
        y - 0.085,
        xerr=[
            np.maximum(null_mu - null_lo, 0),
            np.maximum(null_hi - null_mu, 0),
        ],
        fmt="D",
        markersize=5.8,
        color="#777777",
        ecolor="#B8B8B8",
        elinewidth=2.4,
        capsize=2.8,
        capthick=1.00,
        zorder=2,
    )

    ax.errorbar(
        obs,
        y + 0.085,
        xerr=[
            np.maximum(obs - obs_lo, 0),
            np.maximum(obs_hi - obs, 0),
        ],
        fmt="o",
        markersize=6.2,
        color="black",
        ecolor="black",
        elinewidth=1.8,
        capsize=2.8,
        capthick=1.00,
        zorder=4,
    )

    ax.axvline(
        0,
        color="black",
        linestyle="--",
        linewidth=1.00,
        zorder=0,
    )

    ax.set_yticks(y)
    ax.set_yticklabels(sub["corr_label"].tolist(), fontsize=12.2)
    ax.yaxis.tick_right()
    ax.yaxis.set_label_position("right")

    ax.tick_params(
        axis="y",
        labelsize=12.2,
        pad=4.0,
        length=3.4,
        width=0.95,
        left=False,
        right=True,
    )

    ax.set_xlabel("Vaccination correlation", fontsize=14.0, labelpad=4)
    ax.set_ylabel("")

    ax.set_xlim(-0.17, 0.36)
    ax.set_xticks([-0.15, 0.00, 0.15, 0.30])

    ax.tick_params(
        axis="x",
        labelsize=12.0,
        pad=3.0,
        length=3.4,
        width=0.95,
    )

    ax.xaxis.grid(True, color="#E1E1E1", linewidth=0.65, zorder=0)
    ax.yaxis.grid(False)

    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(1.05)
        ax.spines[side].set_color("#555555")

    ax.margins(y=0.12)

    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            color="black",
            markerfacecolor="black",
            markersize=6.2,
            linestyle="None",
            label="Observed",
        ),
        Line2D(
            [0], [0],
            marker="D",
            color="#777777",
            markerfacecolor="#777777",
            markersize=5.8,
            linestyle="None",
            label="Shuffled null",
        ),
    ]

    legend = ax.legend(
        handles=legend_handles,
        loc="lower right",
        bbox_to_anchor=(0.985, 0.045),
        frameon=True,
        fancybox=False,
        framealpha=0.96,
        facecolor="white",
        edgecolor="#555555",
        fontsize=10.5,
        handletextpad=0.35,
        borderaxespad=0.0,
        borderpad=0.35,
        labelspacing=0.25,
    )
    legend.get_frame().set_linewidth(0.85)
    legend.set_zorder(20)


# ============================================================
# Main figure
# ============================================================

def make_fig4_from_minimal(plotdata: pd.DataFrame) -> None:
    fig = plt.figure(figsize=(17.2, 17.0))

    gs = GridSpec(
        3,
        2,
        figure=fig,
        left=0.055,
        right=0.990,
        top=0.965,
        bottom=0.060,
        hspace=0.28,
        wspace=0.155,
    )

    # B. Work
    ax_B = fig.add_subplot(gs[0, 0])
    draw_binned_density(
        ax_B,
        plotdata,
        panel="B",
        variable="work",
        title="B. Work",
        xlab="Local vaccination fraction",
    )
    plot_work_composition_inset(ax_B, plotdata)

    # C. Education
    ax_C = fig.add_subplot(gs[0, 1])
    draw_binned_density(
        ax_C,
        plotdata,
        panel="C",
        variable="education",
        title="C. Education",
        xlab="Local vaccination fraction",
    )
    add_vaccination_status_legend_to_education(ax_C)

    # D. Household
    plot_household_panel_binned(fig, gs[1, 0], plotdata, panel_title="D. Household")

    # E. Family
    ax_E = fig.add_subplot(gs[1, 1])
    draw_binned_density(
        ax_E,
        plotdata,
        panel="E",
        variable="family",
        title="E. Family",
        xlab="Local vaccination fraction",
    )
    plot_family_correlation_inset(ax_E, plotdata)

    # F. Mean across layers
    ax_F = fig.add_subplot(gs[2, 0])
    draw_binned_density(
        ax_F,
        plotdata,
        panel="F",
        variable="mean_across_layers",
        title="F. Mean across layers",
        xlab="Mean vaccination coverage across layers",
        shade=True,
    )

    # G. Variance across layers
    ax_G = fig.add_subplot(gs[2, 1])
    draw_binned_density(
        ax_G,
        plotdata,
        panel="G",
        variable="variance_across_layers",
        title="G. Variance across layers",
        xlab="Variance of vaccination coverage across layers",
        shade=True,
    )

    fig.savefig(OUT_PDF, bbox_inches="tight")
    fig.savefig(OUT_PNG, dpi=600, bbox_inches="tight")
    fig.savefig(OUT_SVG, bbox_inches="tight")
    plt.close(fig)

    print("Saved:")
    print(f"  {OUT_PDF}")
    print(f"  {OUT_PNG}")
    print(f"  {OUT_SVG}")


def main() -> None:
    if not MINIMAL_FILE.exists():
        raise FileNotFoundError(
            f"Could not find minimal plotting data:\n  {MINIMAL_FILE}\n\n"
            "Put fig4_minimal_plotdata.parquet in the same folder as this script."
        )

    print("Reading minimal plotting data:")
    print(f"  {MINIMAL_FILE}")

    plotdata = pd.read_parquet(MINIMAL_FILE)
    make_fig4_from_minimal(plotdata)


if __name__ == "__main__":
    main()

Reading minimal plotting data:
  /home/karimia1/social_fabric_vacc/github/fig4_plotdata.parquet
Saved:
  /home/karimia1/social_fabric_vacc/github/figure4_BG.pdf
  /home/karimia1/social_fabric_vacc/github/figure4_BG.png
  /home/karimia1/social_fabric_vacc/github/figure4_BG.svg


fig4.ipynb	       figure4_BG.pdf  figure5.png
fig4.py		       figure4_BG.png  figure5.svg
fig4_plotdata.parquet  figure4_BG.svg  figure5_plotdata.parquet
fig5.ipynb	       figure5.pdf     plotdata_summary_kinship_types.parquet
